# server

> `POST` one image or several to `/ocr` and get back each text fragment with its box and confidence.

In [ ]:
#| default_exp server

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import re
import sys
import unicodedata
from pathlib import Path

import numpy as np


def _ensure_repo_on_path() -> Path:
    """Find the checkout that contains the MAX OCR modules."""
    starts = [Path.cwd()]
    file = globals().get("__file__")
    if file:
        starts.insert(0, Path(file).resolve().parent)
    for start in starts:
        for parent in (start, *start.parents):
            if (parent / "ocr.py").exists() and (parent / "model.py").exists():
                root = str(parent)
                if root not in sys.path:
                    sys.path.insert(0, root)
                return parent
    raise FileNotFoundError("solid_lens repo root (ocr.py) was not found")


_ensure_repo_on_path()

from model import compile_detector, compile_recognizer
from ocr import ROOT, detection_hw, load_charset, load_image, read_page

In [ ]:
#| export
def normalize_text(text: str) -> str:
    """Comparison key: compatibility-normalized, casefolded, single-spaced."""
    folded = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", folded).strip()


def box_info(polygon) -> dict:
    """Axis-aligned box and the source polygon, in image pixels."""
    points = np.asarray(polygon, dtype=np.float64).reshape(-1, 2)
    x0, y0 = points.min(axis=0)
    x1, y1 = points.max(axis=0)
    return {
        "x": float(x0),
        "y": float(y0),
        "width": float(x1 - x0),
        "height": float(y1 - y0),
        "polygon": [[float(x), float(y)] for x, y in points],
    }


def _quad(left: int, top: int, right: int, bottom: int) -> list[list[int]]:
    return [[left, top], [right, top], [right, bottom], [left, bottom]]

In [ ]:
#| export
class SolidLensOCR:
    """PP-OCRv6 medium detector and recognizer.

    The recognizer is compiled once. The detector is compiled for the
    PaddleOCR input size of each page and reused for that size.
    """

    def __init__(self) -> None:
        self._detectors: dict[tuple[int, int], object] = {}
        self._recognizer = None
        self._charset = None

    def _recognizer_and_charset(self):
        if self._recognizer is None:
            self._charset = load_charset(ROOT / "models" / "ppocrv6_dict.txt", add_space=True)
            rec_cache = ROOT / "models" / "native_medium_rec_48x320.mef"
            self._recognizer = compile_recognizer(cache_dir=rec_cache)
        return self._recognizer, self._charset

    def _detector_for(self, height: int, width: int):
        shape = detection_hw(height, width)
        detector = self._detectors.get(shape)
        if detector is None:
            det_h, det_w = shape
            cache = ROOT / "models" / f"native_medium_det_{det_h}x{det_w}.mef"
            detector = compile_detector(height=det_h, width=det_w, cache_dir=cache)
            self._detectors[shape] = detector
        return detector

    def predict(self, image_path: str) -> list[dict]:
        """One page result with `dt_polys`, `rec_texts`, and `rec_scores`.

        Polygons are the four corners of each detected line. `rec_scores` are
        the detector confidences for those lines.
        """
        image = load_image(Path(image_path))
        recognizer, charset = self._recognizer_and_charset()
        detector = self._detector_for(*image.shape[:2])
        lines = read_page(image, detector, recognizer, charset)
        polygons, texts, scores = [], [], []
        for text, score, left, top, right, bottom in lines:
            polygons.append(_quad(left, top, right, bottom))
            texts.append(text)
            scores.append(float(score))
        return [{
            "dt_polys": polygons,
            "rec_polys": polygons,
            "rec_texts": texts,
            "rec_scores": scores,
        }]


ocr = SolidLensOCR()


def extract_ocr_items(image_path) -> list[dict]:
    """Run OCR and return text fragments with boxes and confidence."""
    results = ocr.predict(str(image_path))
    items = []

    for result in results:
        data = result
        polygons = data.get("dt_polys") or data.get("rec_polys")
        texts = data.get("rec_texts", [])
        scores = data.get("rec_scores", [])

        if polygons is None:
            raise RuntimeError("No OCR polygons were found in the result.")

        for polygon, text, score in zip(polygons, texts, scores):
            text = str(text).strip()
            if not text:
                continue
            items.append({
                "text": text,
                "norm": normalize_text(text),
                "score": float(score),
                "box": box_info(polygon),
            })

    return items

In [ ]:
#| export
import tempfile

from fasthtml.common import HTTPException, UploadFile, fast_app

app, rt = fast_app(
    default_hdrs=False,
    pico=False,
    htmx=False,
    surreal=False,
    canonical=False,
    secret_key="solid-lens",
)


def _uploaded(form) -> list[UploadFile]:
    files = []
    for key in ("images", "image"):
        files.extend(form.getlist(key))
    return [item for item in files if getattr(item, "filename", None)]


@rt("/")
def get():
    return {
        "post": "/ocr",
        "content_type": "multipart/form-data",
        "fields": ["images", "image"],
        "returns": {"results": [{"name": "file.png", "items": ["text, norm, score, box"]}]},
    }


@rt("/ocr", methods=["POST"])
async def post(req):
    """Read one or more uploaded images and return their OCR items."""
    uploads = _uploaded(await req.form())
    if not uploads:
        raise HTTPException(400, "Send one or more files in the images or image field.")
    results = []
    with tempfile.TemporaryDirectory() as tmp:
        folder = Path(tmp)
        for index, upload in enumerate(uploads):
            name = Path(upload.filename).name
            dest = folder / f"{index}-{name}"
            dest.write_bytes(await upload.read())
            results.append({"name": name, "items": extract_ocr_items(dest)})
    return {"results": results}


def main() -> None:
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=5001)


if __name__ == "__main__":
    main()

In [ ]:
items = extract_ocr_items(_ensure_repo_on_path() / "images" / "image.png")
len(items), items[0] if items else None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()